In [212]:
#imports and installs

%pip install pandas

import math
import pandas as pd
from IPython.display import display, Markdown
import time
import random

Note: you may need to restart the kernel to use updated packages.


# Questão A

## funcoes do dataframe

In [ ]:
def format_scientific(value, precision=4):
    if value == 0:
        return "0"
    
    if abs(value) < 0.01:
        formatted = f"{value:.{precision}e}"
        
        base, exponent = formatted.split("e")
        
        #exponent = int(exponent)
        return f"{base} x 10^{exponent}"
    else:
        return f"{value:.{precision * 2}g}"

def get_numerical_results_table(numerical_data):
    return pd.DataFrame({
        "Bisection": numerical_data["bisection"],
        "False Position": numerical_data["false_position"],
        "Fixed-Point": numerical_data["fixed_point"],
        "Newton": numerical_data["newton"],
        "Secant": numerical_data["secant"]
    }, index=[
        "Initial Data",
        "x̄",
        "f(x̄)",
        "Error in x",
        "Number of Iterations"
    ])

def get_computational_effort_table(effort_data):
    return pd.DataFrame({
        "Bisection": effort_data["bisection"],
        "False Position": effort_data["false_position"],
        "Fixed-Point": effort_data["fixed_point"],
        "Newton": effort_data["newton"],
        "Secant": effort_data["secant"]
    }, index=[
        "Operations per Iteration",
        "Operation Complexity",
        "Logical Decisions",
        "Function Evaluations per Iteration",
        "Total Number of Iterations"
    ])

def get_execution_time_table(time_data):
    return pd.DataFrame({
        "Bisection": time_data["bisection"],
        "False Position": time_data["false_position"],
        "Fixed-Point": time_data["fixed_point"],
        "Newton": time_data["newton"],
        "Secant": time_data["secant"]
    }, index=[
        "Time per Iteration (ms)",
        "Total Time (ms)"
    ])

def print_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Table 1 – Numerical Results for Root-Finding Methods"))
    display(get_numerical_results_table(numerical_data))

    display(Markdown("## Table 2 – Computational Effort Analysis"))
    display(get_computational_effort_table(effort_data))

    display(Markdown("## Table 3 – Execution Time Analysis"))
    display(get_execution_time_table(time_data))

## bisseccao

In [214]:
def bisection(function, interval, stopping_crit_1, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    episolon = stopping_crit_1

    #other initial values
    x = 0
    iterations = 0
    total_time = 0

    #first verification
    if (b - a) < episolon: #(2)
        x = random.uniform(a, b)

    else:
        #(3)
        iterations = 1
        
        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            fa = function(a)

            #(5)
            x = (a + b)/2
            fx = function(x)
            
            #(6)
            if fa * fx > 0:
                a = x
            
            #(7)
            else:
                b = x

            #(8)
            if abs(b - a) < episolon:
                x = random.uniform(a, b)
                break
            
            #(9)
            iterations += 1
        
        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                        #initial data
        x,                               #x̄
        format_scientific(function(x)),  #f(x̄)
        format_scientific(b - a),        #error
        iterations                       #iterations
    ]

    #effort data
    effort_data = [
        4,                     #operations per iteration
        "O(1)",                #complexity
        3,                     #logical decisions
        2,                     #function evals per iteration
        iterations             #iterations
    ]

    #time data
    time_data = [
        total_time/iterations, #time per iteration
        total_time             #total time
    ]

    return numerical_data, effort_data, time_data


## Posicao Falsa

In [215]:
def false_position(function, interval, stopping_crit_1, stopping_crit_2, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2
    
    #other initial values
    x = 0
    iterations = 0
    total_time = 0

    #(2) first verification
    if (b - a) < episolon_1:
        x = random.uniform(a, b)
    
    elif abs(function(a)) < episolon_2:
        x = a
    
    elif abs(function(b)) < episolon_2:
        x = b

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            fa = function(a)
            fb = function(b)

            #(5)
            x = ((a * fb) - (b * fa))/(fb - fa)
            fx = function(x)
            
            #(6)
            if abs(fx) < episolon_2:
                break
            
            #(7)
            if fa * fx > 0:
                a = x

            #(8)
            else:
                b = x

            #(9)
            if abs(b - a) < episolon_1:
                x = random.uniform(a, b)
                break

            #(10)
            iterations += 1
        
        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                        #initial data
        x,                               #x̄
        format_scientific(function(x)),  #f(x̄)
        format_scientific(b - a),        #error
        iterations                       #iterations
    ]

    #effort data
    effort_data = [
        6,                     #operations per iteration
        "O(1)",                #complexity
        6,                     #logical decisions
        3,                     #function evals per iteration
        iterations             #iterations
    ]

    #time data
    time_data = [
        total_time/iterations, #time per iteration
        total_time             #total time
    ]

    return numerical_data, effort_data, time_data

## MPF

In [216]:
def fixed_point(function, iteration_function, interval, stopping_crit_1, stopping_crit_2, max_iterations = 100):
    #(1) initial values
    initial_x = (interval[0] + interval[1])/2
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x
    x_1 = 0
    iterations = 0
    total_time = 0
    current_error = 0

    #(2) first verification
    if abs(function(initial_x)) < episolon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            x_1 = iteration_function(x)
            fx_1 = function(x_1)

            #(5)
            current_error = x_1 - x
            if abs(fx_1) < episolon_1 or abs(current_error) < episolon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",               #initial data
        x,                                 #x̄
        format_scientific(function(x)),    #f(x̄)
        format_scientific(current_error),  #error
        iterations                         #iterations
    ]

    #effort data
    effort_data = [
        1,                     #operations per iteration
        "O(1)",                #complexity
        2,                     #logical decisions
        2,                     #function evals per iteration
        iterations             #iterations
    ]

    #time data
    time_data = [
        total_time/iterations, #time per iteration
        total_time             #total time
    ]

    return numerical_data, effort_data, time_data

## Newton

In [217]:
def get_derivative_function(function):
    def derivative(x, h=1e-8):
        return (function(x + h) - function(x - h)) / (2 * h)
    
    return derivative

def newton(function, interval, stopping_crit_1, stopping_crit_2, max_iterations = 100):
    derivative_function = get_derivative_function(function)

    #(1) initial values
    initial_x = (interval[0] + interval[1])/2
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x
    x_1 = 0
    iterations = 0
    total_time = 0
    current_error = 0

    #(2) first verification
    if abs(function(initial_x)) < episolon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            x_1 = x - (function(x)/derivative_function(x))
            fx_1 = function(x_1)

            #(5)
            current_error = x_1 - x
            if abs(fx_1) < episolon_1 or abs(current_error) < episolon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                 #initial data
        x,                                   #x̄
        format_scientific(function(x)),      #f(x̄)
        format_scientific(current_error),    #error
        iterations                           #iterations
    ]

    #effort data
    effort_data = [
        1,                     #operations per iteration
        "O(1)",                #complexity
        2,                     #logical decisions
        2,                     #function evals per iteration
        iterations             #iterations
    ]

    #time data
    time_data = [
        total_time/iterations, #time per iteration
        total_time             #total time
    ]

    return numerical_data, effort_data, time_data

## Secant

In [218]:
def secant(function, interval, stopping_crit_1, stopping_crit_2, max_iterations = 100):
    #(1) initial values
    initial_x_0 = interval[0]
    initial_x_1 = interval[1]
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x_1
    x_0 = initial_x_0
    x_1 = initial_x_1
    iterations = 0
    total_time = 0
    current_error = 0

    #(2) first verification
    if abs(function(x_0)) < episolon_1:
        x = x_0
    
    #(3) second verification
    elif abs(function(x_1)) < episolon_1 or abs(x_1 - x_0) < episolon_2:
        x = x_1
        
    else:
        #(4)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            fx_0 = function(x_0)
            fx_1 = function(x_1)

            #(5)
            x_2 = x_1 - ((fx_1/(fx_1 - fx_0)) * (x_1 - x_0))
            fx_2 = function(x_2)

            #(6)
            current_error = x_2 - x_1
            if abs(fx_2) < episolon_1 or abs(current_error) < episolon_2:
                x = x_2
                break
            
            #(7)
            x_0 = x_1
            x_1 = x_2

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x_0}; X1: {initial_x_1}",   #initial data
        x,                                          #x̄
        format_scientific(function(x)),             #f(x̄)
        format_scientific(current_error),           #error
        iterations                                  #iterations
    ]

    #effort data
    effort_data = [
        1,                     #operations per iteration
        "O(1)",                #complexity
        2,                     #logical decisions
        2,                     #function evals per iteration
        iterations             #iterations
    ]

    #time data
    time_data = [
        total_time/iterations, #time per iteration
        total_time             #total time
    ]

    return numerical_data, effort_data, time_data

## datas

In [219]:
numerical_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

effort_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

time_data = {
    "bisection": [None, None],
    "false_position": [None, None],
    "fixed_point": [None, None],
    "newton": [None, None],
    "secant": [None, None]
}

## Exemplo 18

In [220]:
example_function = lambda x: (math.e**(-x**2)) - math.cos(x)
fixed_point_iteration_function = lambda x: math.cos(x) - math.e**(-x**2) + x
interval = [1, 2]
stopping_crit_1 = stopping_crit_2 = 10**-4

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit_1)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit_1, stopping_crit_2)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, interval, stopping_crit_1, stopping_crit_2)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, interval, stopping_crit_1, stopping_crit_2)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, interval, stopping_crit_1, stopping_crit_2)

print_tables(numerical_data, effort_data, time_data)

ValueError: Invalid format specifier '.6.0g' for object of type 'float'